# Assignment 1 — QANet (High-EM Training Notebook)

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. This notebook is configured for **full-data training** to maximize dev F1 / EM.

> Before training, switch Colab to **GPU runtime**. Full-data QANet is too memory-heavy for a normal CPU runtime.

> If you only want a quick smoke test, switch Section 1 and Section 2 back to `download_mini()` and `train-mini.json`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026_fixed"


In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en_core_web_sm


---
## Section 0 — Environment Setup

Mount Google Drive, install dependencies, set the working directory, and then force-sync the repo so Colab does not keep using an older checkout or cached module state.


In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('sys.path[0]:', sys.path[0])


---
## Section 0.5 — Force-Sync the Repo and Verify New Files Exist

> Run this after mounting Drive and setting `PROJECT_ROOT`. This avoids a common Colab problem where the notebook is still using an older checkout or stale module cache.


In [ ]:
!git -C {PROJECT_ROOT} pull origin main
!git -C {PROJECT_ROOT} rev-parse --short HEAD
!ls {PROJECT_ROOT}/Tools


> You should see `data_diagnostics.py` inside `Tools/`. If it is missing, restart the runtime, re-mount Drive, and run the sync cell again before continuing.


---
## Section 1 — Download Full Data *(one-time setup)*

Downloads the full SQuAD v1.1 train/dev set and the full GloVe 840B 300d vectors into `_data/`.

> This is the recommended setup if your goal is to improve EM substantially. The mini dataset is useful only for pipeline smoke tests.


In [ ]:
from Tools.download import download

download(data_dir="_data")


---
## Section 2 — Preprocess Full Data *(rebuild from scratch for a real retrain)*

Tokenises the full SQuAD JSON files, builds word/char vocabularies from the full GloVe file, and writes padded index tensors to `_data/`.

> If you updated the repo code, you should **delete the old preprocessing outputs and rebuild them** before training again. This is especially important after embedding / vocab fixes.


In [ ]:
from pathlib import Path
import shutil
from Tools.preproc import preprocess

for path in [
    '_data/train.npz', '_data/dev.npz',
    '_data/word_emb.json', '_data/char_emb.json',
    '_data/word2idx.json', '_data/char2idx.json',
    '_data/train_eval.json', '_data/dev_eval.json', '_data/dev_meta.json',
]:
    p = Path(path)
    if p.exists():
        p.unlink()

preprocess(
    train_file='_data/squad/train-v1.1.json',
    dev_file='_data/squad/dev-v1.1.json',
    glove_word_file='_data/glove/glove.840B.300d.txt',
    target_dir='_data',
    para_limit=400,
    ques_limit=50,
)


---
## Section 2.2 — Inspect Preprocessing Statistics

> This helps confirm that the rebuilt `_data/` cache is using the latest preprocessing logic and that embedding coverage is reasonable.


In [ ]:
import json

with open('_data/preprocess_stats.json', 'r') as f:
    preprocess_stats = json.load(f)

print(json.dumps(preprocess_stats, indent=2))


---
## Section 2.25 — Check Label Upper Bound

> This decodes the stored gold `(y1, y2)` spans back into text and compares them against the official answers. If this upper bound is low, then the preprocessing labels are misaligned and long training will fail.


In [ ]:
from importlib import invalidate_caches
invalidate_caches()

from Tools.data_diagnostics import label_upper_bound

train_label_upper = label_upper_bound('_data/train.npz', '_data/train_eval.json', limit=2000)
dev_label_upper = label_upper_bound('_data/dev.npz', '_data/dev_eval.json', limit=2000)

print('TRAIN LABEL UPPER:', train_label_upper)
print('DEV LABEL UPPER:', dev_label_upper)


---
## Section 2.3 — Optional Tiny-Subset Overfit Check

> Run this before a long full-data training job. If the model cannot memorize a tiny subset, then there is still a pipeline / preprocessing / implementation problem and a long run is likely to waste time.


In [ ]:
from TrainTools.overfit_check import overfit_check

overfit_results = overfit_check(
    train_npz='_data/train.npz',
    word_emb_json='_data/word_emb.json',
    char_emb_json='_data/char_emb.json',
    train_eval_json='_data/train_eval.json',
    subset_size=64,
    save_dir='_overfit_check',
    log_dir='_overfit_log',
    num_steps=400,
    checkpoint=100,
    batch_size=16,
    learning_rate=3e-3,
)

print(overfit_results['eval_metrics'])


---
## Section 2.5 — Clear Old Checkpoints Before a Fresh Full Retrain

> Run this once before starting a new full-data training run, so you do not accidentally evaluate stale checkpoints.


In [ ]:
from pathlib import Path
import shutil

for path in ['_model_full', '_log_full']:
    p = Path(path)
    if p.exists():
        shutil.rmtree(p)
        print(f'removed {p}')

Path('_model_full').mkdir(exist_ok=True)
Path('_log_full').mkdir(exist_ok=True)
print('fresh output directories ready')


---
## Section 3 — Train for Higher EM

Trains QANet on the full SQuAD v1.1 training set and saves the **best checkpoint by dev EM/F1** to `_model_full/model.pt`.

> First run the preprocessing stats cell, the label upper-bound check, and the tiny-subset overfit check. If either sanity check fails, stop and debug before launching the long run.

> For the current high-memory Colab setup, use an aggressive configuration first: `batch_size=32`, `grad_accum_steps=1`, and `num_steps=20000`. If that unexpectedly runs out of memory, fall back to `batch_size=16`.


In [ ]:
from TrainTools.train import train

results = train(
    train_npz       = '_data/train.npz',
    dev_npz         = '_data/dev.npz',
    word_emb_json   = '_data/word_emb.json',
    char_emb_json   = '_data/char_emb.json',
    train_eval_json = '_data/train_eval.json',
    dev_eval_json   = '_data/dev_eval.json',
    save_dir        = '_model_full',
    log_dir         = '_log_full',
    num_steps       = 20000,
    checkpoint      = 2000,
    batch_size      = 32,
    seed            = 42,
    grad_accum_steps = 1,
    optimizer_name  = 'adam',
    scheduler_name  = 'cosine',
    learning_rate   = 1e-3,
    loss_name       = 'qa_nll',
    norm_name       = 'layer_norm',
    test_num_batches = -1,
    max_answer_len  = 30,
)

print(f"Best F1: {results['best_f1']:.3f}%  |  Best EM: {results['best_em']:.3f}%")


---
## Section 4 — Evaluate Best Checkpoint

Loads the saved best checkpoint and runs inference on the **full dev set**.


In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz          = '_data/dev.npz',
    word_emb_json    = '_data/word_emb.json',
    char_emb_json    = '_data/char_emb.json',
    dev_eval_json    = '_data/dev_eval.json',
    save_dir         = '_model_full',
    log_dir          = '_log_full',
    ckpt_name        = 'model.pt',
    test_num_batches = -1,
    max_answer_len   = 30,
)

print(f"F1: {metrics['f1']:.3f}%  |  EM: {metrics['exact_match']:.3f}%  |  Loss: {metrics['loss']:.6f}")
